[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghodsizadeh/tehran-stocks/blob/master/Example/Market_Screening_and_Charts.ipynb)
[![Documentation](https://img.shields.io/badge/Docs-%D9%85%D8%B3%D8%AA%D9%86%D8%AF%D8%A7%D8%AA%20%D9%81%D8%A7%D8%B1%D8%B3%DB%8C-0d9488.svg)](https://ghodsizadeh.github.io/tehran-stocks/)
[![GitHub stars](https://img.shields.io/github/stars/ghodsizadeh/tehran-stocks?style=social)](https://github.com/ghodsizadeh/tehran-stocks)

# Tehran Stocks v3 — Full-Market Screening, Async Watchlists & Interactive Plotly Charts

Screen all ~3,800 TSE instruments in **2 HTTP calls**, compare TEDPIX vs. Equal-Weighted breadth, fetch a multi-stock portfolio concurrently with `AsyncClient`, and plot interactive candlestick + money-flow charts with **Plotly** (replacing legacy `cufflinks`).

> **Note:** All TSETMC prices and trade values are in **Iranian Rials (IRR)** (`1 Toman = 10 IRR`; `1 Billion Toman = 1e10 IRR`).

## 1. Installation & Imports

In [ ]:
%pip install -q "tehran-stocks>=3,<4" plotly

In [ ]:
import asyncio
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tehran_stocks as ts

## 2. Market Breadth: TEDPIX vs. Equal-Weighted Index & Top Sectors

In [ ]:
with ts.Client() as client:
    tse = client.market.overview(flow=1)
    ifb = client.market.overview(flow=2)
    sectors = client.market.sectors()
    effects = client.market.index_effect("TEDPIX", flow=1, top=5)

print(f"TSE State:          {tse.market_state}")
print(f"TEDPIX:             {tse.index_value:,.0f} ({tse.index_change:+,.0f})")
print(f"Equal-Weighted:     {tse.equal_weighted_index:,.0f} ({tse.equal_weighted_change:+,.0f})")
print(f"TSE Trade Value:    {tse.trade_value / 1e10:,.0f} Billion Toman")
print(f"Farabourse Index:   {ifb.index_value:,.0f} ({ifb.index_change:+,.0f})")

print("\nTop 5 TEDPIX Index Movers:")
for item in effects:
    print(f"  {item.symbol:10s} | effect={item.effect:+.1f} | close={item.close:,.0f} IRR")

top_sectors = sorted(sectors, key=lambda s: s.trade_value, reverse=True)[:10]
pd.DataFrame(
    [
        {
            "code": s.code,
            "sector": s.name,
            "trade_value_bt": round(s.trade_value / 1e10, 1),
            "trade_count": s.trade_count,
        }
        for s in top_sectors
    ]
)

## 3. Full-Market Screener in 2 HTTP Calls

`client.market.watch()` returns all ~3,800 live instruments (with 5-level order books), and `client.client_type.all()` returns live individual/institutional flows for the entire market — one request each.

In [ ]:
with ts.Client() as client:
    watch_items = client.market.watch()
    flow_items = client.client_type.all()

flow_by_code = {f.ins_code: f for f in flow_items}

candidates = []
for item in watch_items:
    flow = flow_by_code.get(item.ins_code)
    if flow is None:
        continue

    top = item.order_book[0] if item.order_book else None
    is_buy_queue = (
        top is not None
        and item.price_max_allowed is not None
        and top.bid_price >= item.price_max_allowed
        and top.ask_volume == 0
    )

    if (
        item.value >= 50_000_000_000  # >= 5 Billion Toman trade value
        and flow.real_money_flow > 0
        and flow.buyer_power >= 1.5
        and not is_buy_queue
    ):
        candidates.append(
            {
                "symbol": item.symbol,
                "name": item.name,
                "sector_code": item.sector_code,
                "last": item.last,
                "change_pct": round(item.change_pct, 2),
                "value_bt": round(item.value / 1e10, 1),
                "real_flow_bt": round(flow.real_money_flow / 1e10, 2),
                "buyer_power": round(flow.buyer_power, 2),
            }
        )

screen_df = (
    pd.DataFrame(candidates)
    .sort_values("real_flow_bt", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
screen_df

## 4. Parallel Multi-Stock Portfolio Fetch with `AsyncClient`

Fetch adjusted histories for an entire watchlist concurrently with bounded concurrency (`max_concurrency=6`).

In [ ]:
WATCHLIST = ["فولاد", "فملی", "کگل", "شپنا", "خودرو"]


async def fetch_watchlist_closes(symbols: list[str], days: int = 200) -> pd.DataFrame:
    async with ts.AsyncClient(max_concurrency=6) as client:
        histories = await asyncio.gather(
            *(client.prices.history(sym, days=days, adjusted=True) for sym in symbols)
        )
    series_map = {
        sym: ts.to_dataframe(bars, jalali=True, index_by_date=True)["close"]
        for sym, bars in zip(symbols, histories)
    }
    return pd.DataFrame(series_map).dropna(how="all")


# Notebooks (Jupyter / Colab) already run an event loop, so `await` works directly.
closes = await fetch_watchlist_closes(WATCHLIST, days=200)

returns_pct = ((closes.iloc[-1] / closes.iloc[0]) - 1) * 100
print("=== 200-Day Adjusted Return (%) ===")
print(returns_pct.sort_values(ascending=False).round(1))

print("\n=== Daily Return Correlation Matrix ===")
closes.pct_change().corr().round(2)

## 5. Interactive Plotly Candlestick + SMA/EMA + Volume Chart

Modern replacement for legacy `cufflinks`: native `plotly.graph_objects` candlesticks with Jalali hover dates.

In [ ]:
CHART_SYMBOL = "کگل"

with ts.Client() as client:
    bars = client.prices.history(CHART_SYMBOL, days=180, adjusted=True)

df = ts.to_dataframe(bars, jalali=True, index_by_date=True)
df["jdate_str"] = df["date"].jalali.to_jalali().jalali.strftime("%Y-%m-%d")
df["sma_20"] = df["close"].rolling(20).mean()
df["ema_20"] = df["close"].ewm(span=20, adjust=False).mean()

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.75, 0.25],
    subplot_titles=(f"{CHART_SYMBOL} — Adjusted Candlestick (IRR)", "Volume"),
)

fig.add_trace(
    go.Candlestick(
        x=df["jdate_str"],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=df["jdate_str"], y=df["sma_20"], name="SMA 20", line={"width": 1.4}),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=df["jdate_str"], y=df["ema_20"], name="EMA 20", line={"width": 1.4, "dash": "dot"}),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=df["jdate_str"], y=df["volume"], name="Volume", opacity=0.6),
    row=2,
    col=1,
)

fig.update_layout(
    height=700,
    xaxis_rangeslider_visible=False,
    xaxis2_type="category",
    xaxis_type="category",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

## 6. Bonus: CLI (`tse`) & AI Assistant (`MCP`) One-Liners

You can run the same screener from the terminal or connect your AI assistant (Claude, Cursor, Codex, Antigravity) with zero extra code:

In [ ]:
!tse market
!tse screen --sort money_flow --limit 10